In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import yfinance as yf

In [3]:
ticker = '^NSEI'
start_date = '2014-01-01'
end_date = '2025-01-01'

df = yf.download(ticker,start_date,end_date)
df

/tmp/ipykernel_2168/1502302891.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker,start_date,end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2014-01-02,6221.149902,6358.299805,6211.299805,6301.250000,158100
2014-01-03,6211.149902,6221.700195,6171.250000,6194.549805,139000
2014-01-06,6191.450195,6224.700195,6170.250000,6220.850098,118300
2014-01-07,6162.250000,6221.500000,6144.750000,6203.899902,138600
2014-01-08,6174.600098,6192.100098,6160.350098,6178.049805,146900
...,...,...,...,...,...
2024-12-24,23727.650391,23867.650391,23685.150391,23769.099609,177700
2024-12-26,23750.199219,23854.500000,23653.599609,23775.800781,177700


**A1 - STATIONARITY**

In [4]:
from statsmodels.tsa.stattools import adfuller

In [5]:
result = adfuller(df['Close']['^NSEI'])
print(result[1])

0.97864667584301


In [6]:
daily_returns_matrix = df['Close']['^NSEI'].pct_change().dropna()
daily_returns_matrix.dropna(inplace=True)
result = adfuller(daily_returns_matrix)
print(result[1])

7.245295456352895e-27


**A2 - LOOK AHEAD BIAS**

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
returns = df['Close']['^NSEI'].pct_change().dropna()
returns = returns.reset_index(drop=True)


df_returns = pd.DataFrame()
df_returns['returns'] = returns.values
df_returns['lag1'] = df_returns['returns'].shift(1)
df_returns['target'] = (df_returns['returns'] > 0).astype(int)
df_returns.dropna(inplace=True)

df_returns

,returns,lag1,target
1,-0.003172,-0.001607,0
2,-0.004716,-0.003172,0
3,0.002004,-0.004716,1
4,-0.001012,0.002004,0
5,0.000503,-0.001012,1
...,...,...,...
2693,-0.001086,0.007035,0
2694,0.000950,-0.001086,1
2695,0.002661,0.000950,1
2696,-0.007076,0.002661,0


**A3 - WALK-FORWARD VALIDATION**

In [9]:
from sklearn.linear_model import LogisticRegression

In [10]:
X = df_returns[['lag1']].values
y = df_returns['target'].values

# Split 1: shuffled (wrong way)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
model = LogisticRegression()
model.fit(X_train, y_train)
print("Shuffled accuracy:", model.score(X_test, y_test))

# Split 2: time-ordered (correct way)
split = int(len(X) * 0.8)
X_train2, X_test2 = X[:split], X[split:]
y_train2, y_test2 = y[:split], y[split:]
model2 = LogisticRegression()
model2.fit(X_train2, y_train2)
print("Time-ordered accuracy:", model2.score(X_test2, y_test2))


Shuffled accuracy: 0.5425925925925926
Time-ordered accuracy: 0.562962962962963


In [11]:
accuracies = []
for i in range(1500, len(X), 50):
  X_train = X[:i]
  y_train = y[:i]
  X_test = X[i:i+50]
  y_test = y[i:i+50]
  model = LogisticRegression()
  model.fit(X_train, y_train)
  model.predict(X_test)
  accuracies.append(model.score(X_test, y_test))

print(np.mean(accuracies))

0.557677304964539


**A4 - OVERFITTING IN FINANCE**

In [12]:
from sklearn.tree import DecisionTreeClassifier

X = df_returns[['lag1']]
y = df_returns['target']
X_train = X.iloc[0:1501]
y_train = y.iloc[0:1501]

model = DecisionTreeClassifier()
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 1.0
Test accuracy: 0.5376254180602007


In [13]:
model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 0.5489673550966022
Test accuracy: 0.5451505016722408


**FEATURE ENGINEERING**

In [14]:
df_returns['lag5'] = df_returns['returns'].shift(5)
df_returns.dropna(inplace=True)
df_returns['rolling mean'] = df_returns['returns'].rolling(window=10).mean().shift(1)
df_returns['rolling volatility'] = df_returns['returns'].rolling(window=10).std().shift(1)
df_returns.dropna(inplace=True)
df_returns.head(15)

,returns,lag1,target,lag5,rolling mean,rolling volatility
16,-0.020888,-0.012434,0,0.006755,0.001570,0.009014
17,-0.001565,-0.020888,0,0.001562,-0.002160,0.009867
18,-0.000979,-0.001565,0,0.003983,-0.001824,0.009819
19,-0.007606,-0.000979,0,0.001057,-0.003188,0.008432
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471
21,-0.014402,0.002601,0,-0.020888,-0.002751,0.008487
22,-0.000150,-0.014402,0,-0.001565,-0.004867,0.008491
23,0.003583,-0.000150,1,-0.000979,-0.005038,0.008363
24,0.002308,0.003583,1,-0.007606,-0.005078,0.008316
25,0.004456,0.002308,1,0.002601,-0.004953,0.008427


In [15]:
volume = df['Volume']['^NSEI'].reset_index(drop=True)
df_returns['rolling volume 5'] = volume.rolling(5).mean().shift(1)
df_returns['rolling volume 10'] = volume.rolling(10).mean().shift(1)
df_returns['volume ratio'] = (volume/volume.rolling(10).mean()).shift(1)
df_returns = df_returns.reindex(df_returns.index)
df_returns

,returns,lag1,target,lag5,rolling mean,rolling volatility,rolling volume 5,rolling volume 10,volume ratio
16,-0.020888,-0.012434,0,0.006755,0.001570,0.009014,137820.0,139350.0,0.861859
17,-0.001565,-0.020888,0,0.001562,-0.002160,0.009867,136340.0,139390.0,1.150011
18,-0.000979,-0.001565,0,0.003983,-0.001824,0.009819,150000.0,144930.0,1.313738
19,-0.007606,-0.000979,0,0.001057,-0.003188,0.008432,158480.0,152320.0,1.208640
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471,160320.0,152400.0,0.962598
...,...,...,...,...,...,...,...,...,...
2693,-0.001086,0.007035,0,-0.013469,-0.003543,0.008063,280320.0,267420.0,1.655448
2694,0.000950,-0.001086,1,-0.005636,-0.003616,0.008034,280760.0,261590.0,0.725563
2695,0.002661,0.000950,1,-0.010213,-0.003650,0.008012,263320.0,253260.0,0.701650
2696,-0.007076,0.002661,0,-0.015206,-0.003006,0.008255,251800.0,252300.0,0.704320


**RSI**

In [16]:
delta = df['Close']['^NSEI'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
RSI = RSI.reset_index(drop=True)
df_returns['RSI'] = RSI.shift(1)
df_returns

,returns,lag1,target,lag5,rolling mean,rolling volatility,rolling volume 5,rolling volume 10,volume ratio,RSI
16,-0.020888,-0.012434,0,0.006755,0.001570,0.009014,137820.0,139350.0,0.861859,65.819822
17,-0.001565,-0.020888,0,0.001562,-0.002160,0.009867,136340.0,139390.0,1.150011,57.774090
18,-0.000979,-0.001565,0,0.003983,-0.001824,0.009819,150000.0,144930.0,1.313738,47.747447
19,-0.007606,-0.000979,0,0.001057,-0.003188,0.008432,158480.0,152320.0,1.208640,45.855110
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471,160320.0,152400.0,0.962598,45.874774
...,...,...,...,...,...,...,...,...,...,...
2693,-0.001086,0.007035,0,-0.013469,-0.003543,0.008063,280320.0,267420.0,1.655448,33.254706
2694,0.000950,-0.001086,1,-0.005636,-0.003616,0.008034,280760.0,261590.0,0.725563,32.759206
2695,0.002661,0.000950,1,-0.010213,-0.003650,0.008012,263320.0,253260.0,0.701650,32.011432
2696,-0.007076,0.002661,0,-0.015206,-0.003006,0.008255,251800.0,252300.0,0.704320,23.932179


In [17]:
ma20 = df['Close']['^NSEI'].rolling(20).mean()
std20 = df['Close']['^NSEI'].rolling(20).std()
bb_width = (2 * std20) / ma20
bb_width = bb_width.reset_index(drop=True)
df_returns['bb_width'] = bb_width.shift(1)
df_returns

,returns,lag1,target,lag5,rolling mean,rolling volatility,rolling volume 5,rolling volume 10,volume ratio,RSI,bb_width
16,-0.020888,-0.012434,0,0.006755,0.001570,0.009014,137820.0,139350.0,0.861859,65.819822,NaN
17,-0.001565,-0.020888,0,0.001562,-0.002160,0.009867,136340.0,139390.0,1.150011,57.774090,NaN
18,-0.000979,-0.001565,0,0.003983,-0.001824,0.009819,150000.0,144930.0,1.313738,47.747447,NaN
19,-0.007606,-0.000979,0,0.001057,-0.003188,0.008432,158480.0,152320.0,1.208640,45.855110,NaN
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471,160320.0,152400.0,0.962598,45.874774,0.024019
...,...,...,...,...,...,...,...,...,...,...,...
2693,-0.001086,0.007035,0,-0.013469,-0.003543,0.008063,280320.0,267420.0,1.655448,33.254706,0.025627
2694,0.000950,-0.001086,1,-0.005636,-0.003616,0.008034,280760.0,261590.0,0.725563,32.759206,0.027910
2695,0.002661,0.000950,1,-0.010213,-0.003650,0.008012,263320.0,253260.0,0.701650,32.011432,0.030038
2696,-0.007076,0.002661,0,-0.015206,-0.003006,0.008255,251800.0,252300.0,0.704320,23.932179,0.031827


In [19]:
ema12 = df['Close']['^NSEI'].ewm(span=12).mean()
ema26 = df['Close']['^NSEI'].ewm(span=26).mean()
macd = ema12 - ema26
macd = macd.reset_index(drop=True)
df_returns['macd'] = macd.shift(1)
df_returns.dropna(inplace=True)
df_returns

,returns,lag1,target,lag5,rolling mean,rolling volatility,rolling volume 5,rolling volume 10,volume ratio,RSI,bb_width,macd
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471,160320.0,152400.0,0.962598,45.874774,0.024019,-11.268955
21,-0.014402,0.002601,0,-0.020888,-0.002751,0.008487,177920.0,157870.0,1.318173,42.198091,0.026647,-19.988033
22,-0.000150,-0.014402,0,-0.001565,-0.004867,0.008491,175200.0,155770.0,0.941773,33.062181,0.028419,-25.440478
23,0.003583,-0.000150,1,-0.000979,-0.005038,0.008363,164100.0,157050.0,0.858962,29.920494,0.032490,-35.325680
24,0.002308,0.003583,1,-0.007606,-0.005078,0.008316,163940.0,161210.0,1.137026,19.207059,0.035761,-42.648207
...,...,...,...,...,...,...,...,...,...,...,...,...
2693,-0.001086,0.007035,0,-0.013469,-0.003543,0.008063,280320.0,267420.0,1.655448,33.254706,0.025627,-45.527084
2694,0.000950,-0.001086,1,-0.005636,-0.003616,0.008034,280760.0,261590.0,0.725563,32.759206,0.027910,-82.033872
2695,0.002661,0.000950,1,-0.010213,-0.003650,0.008012,263320.0,253260.0,0.701650,32.011432,0.030038,-111.759223
2696,-0.007076,0.002661,0,-0.015206,-0.003006,0.008255,251800.0,252300.0,0.704320,23.932179,0.031827,-131.975939


In [25]:
df_global = yf.download(['^GSPC', 'USDINR=X'], start='2014-01-01', end='2025-01-01')

/tmp/ipykernel_2168/233716698.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_global = yf.download(['^GSPC', 'USDINR=X'], start='2014-01-01', end='2025-01-01')
[*********************100%***********************]  2 of 2 completed


In [29]:
sp_500_returns = df_global['Close']['^GSPC'].pct_change(fill_method = None)
usdinr_returns = df_global['Close']['USDINR=X'].pct_change(fill_method = None)
sp_500_returns = sp_500_returns.reset_index(drop=True)
usdinr_returns = usdinr_returns.reset_index(drop=True)
df_returns['sp_500_returns'] = sp_500_returns.shift(1)
df_returns['usdinr_returns'] = usdinr_returns.shift(1)
df_returns

,returns,lag1,target,lag5,rolling mean,rolling volatility,rolling volume 5,rolling volume 10,volume ratio,RSI,bb_width,macd,sp_500_returns,usdinr_returns
20,0.002601,-0.007606,1,-0.012434,-0.003917,0.008471,160320.0,152400.0,0.962598,45.874774,0.024019,-11.268955,0.006141,0.010362
21,-0.014402,0.002601,0,-0.020888,-0.002751,0.008487,177920.0,157870.0,1.318173,42.198091,0.026647,-19.988033,-0.010209,-0.015778
22,-0.000150,-0.014402,0,-0.001565,-0.004867,0.008491,175200.0,155770.0,0.941773,33.062181,0.028419,-25.440478,0.011267,0.001411
23,0.003583,-0.000150,1,-0.000979,-0.005038,0.008363,164100.0,157050.0,0.858962,29.920494,0.032490,-35.325680,-0.006465,-0.002529
24,0.002308,0.003583,1,-0.007606,-0.005078,0.008316,163940.0,161210.0,1.137026,19.207059,0.035761,-42.648207,-0.022832,0.003049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2693,-0.001086,0.007035,0,-0.013469,-0.003543,0.008063,280320.0,267420.0,1.655448,33.254706,0.025627,-45.527084,0.010209,-0.000466
2694,0.000950,-0.001086,1,-0.005636,-0.003616,0.008034,280760.0,261590.0,0.725563,32.759206,0.027910,-82.033872,0.003178,0.001206
2695,0.002661,0.000950,1,-0.010213,-0.003650,0.008012,263320.0,253260.0,0.701650,32.011432,0.030038,-111.759223,-0.015731,0.000585
2696,-0.007076,0.002661,0,-0.015206,-0.003006,0.008255,251800.0,252300.0,0.704320,23.932179,0.031827,-131.975939,-0.003435,0.000567
